<a href="https://colab.research.google.com/github/eviemcmahan/ds2002-fa26/blob/main/notebooks/02-sql-databases/2026-09-11%20%E2%80%94%20SQL%20Challenge%20Set%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [82]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [83]:
q1 = q('''
  SELECT t.title, a.name, a.country
  FROM artists a
  LEFT JOIN tracks t ON a.artist_id = t.artist_id
''')
print('Rows:', len(q1))
q1

Rows: 9


,title,name,country
0,Skyline,Nova Waves,US
1,Undertow,Nova Waves,US
2,Coastline,The Blue Ridge,US
3,Foothills,The Blue Ridge,US
4,Ridgeline,The Blue Ridge,US
5,Aurora,Kestrel,UK
6,Nightfall,Kestrel,UK
7,Untitled Demo,Kestrel,UK
8,Sol,Marisol,ES


I used a left join to ensure I didn't lose any tracks that didn't have an `artist_id`, although that isn't a problem in this dataset. I kept in the untitled track since it still has an artist and country. The row count matches with the 9 expected rows.

### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [84]:
q2 = q('''
  SELECT t.genre, ROUND(AVG(t.seconds)/60.0,1) as avg_minutes
  FROM tracks t
  WHERE t.genre IS NOT NULL
  GROUP BY t.genre
  ORDER BY avg_minutes DESC
  LIMIT 1
''')
q2

,genre,avg_minutes
0,Electronic,4.8


I selected genre and calculated the average minutes for each track from the tracks tab, and then grouped by genre and sorted to show the longest average track length. I excluded the untitled track since it's genre is unknown, and having an unknown genre isn't helpful when trying to find the genre with the longest average track length. Either way, Electronic has an average track length of 4.8 minutes, which is longer than the 2.5 minute track length of the untagged track. I included the unplayed tracks since whether or not the track is played doesn't impact its length.

### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [85]:
q3 = q('''
  SELECT p.user, COUNT(p.play_id) as plays, COUNT(DISTINCT p.track_id) as distinct_tracks
  FROM plays p
  GROUP BY p.user
''')
q3

,user,plays,distinct_tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


I counted the plays and distinct tracks of each user, grouped by user. None of the users played the untagged track, but I wouldn't have excluded it even if they did since that is a separate track they are playing. The row count of 4 matches the 4 users that I expected to get as a result.

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [86]:
q4 = q('''
  SELECT t.track_id, t.title, COUNT(p.play_id) as plays
  FROM tracks t
  LEFT JOIN plays p ON t.track_id = p.track_id
  GROUP BY t.track_id
  HAVING plays = 0
''')
q4

,track_id,title,plays
0,17,Ridgeline,0
1,18,Untitled Demo,0


I added a `play_id` count to count up the number of plays for each `track_id` (since tracks are being sorted by `track_id`). This eliminated me having to deal with unplayed tracks as null, since instead they had a play count of 0. I included the untagged track since it is still a real track with a real artist, so I believe it's a valid answer to the question. My answer returned 2 rows, "Ridgeline" and "Untitled Demo," which matches the expected row count.

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [87]:
q5 = q('''
  SELECT a.name, SUM(t.seconds) as seconds_listened, ROUND(SUM(t.seconds)/60.0,1) as minutes_listened
  FROM artists a
  JOIN tracks t ON a.artist_id = t.artist_id
  JOIN plays p ON t.track_id = p.track_id
  GROUP BY a.name
  ORDER BY seconds_listened DESC
''')
q5

,name,seconds_listened,minutes_listened
0,Kestrel,1175,19.6
1,Nova Waves,843,14.1
2,The Blue Ridge,384,6.4
3,Marisol,210,3.5


I used two inner joins to complete this task. I assumed that for each play, the user listened to the entire song. I didn't exclude the untagged track, but via inner join I excluded the unplayed tracks (since they don't have a line in `plays`). This was on purpose since only plays should impact the number of seconds listened to. My results matched the 4 expected rows since there are 4 artists in my data set.

### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [88]:
q6 = q('''
  SELECT t.track_id, t.title, t.genre
  FROM tracks t
  WHERE t.genre is NULL
''')
q6

,track_id,title,genre
0,18,Untitled Demo,None


The untagged track ("Untitled Demo") is missing a genre. `WHERE genre != 'Pop'` would have excluded this track since `WHERE` only returns values that are true, and null `t.genre` values would return unknown. I expected this to return 1 row.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [89]:
q7 = q('''
  SELECT p.played_on, COUNT(p.play_id) as plays, COUNT(DISTINCT p.user) as users
  FROM plays p
  GROUP BY p.played_on
  ORDER BY p.played_on ASC
''')
q7

,played_on,plays,users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


I expected this to return 6 rows, since there are 6 dates within the `plays` table. I grouped by day and counted the number of plays and distinct users. Untagged and unplayed values didn't impact this answer since plays only takes into account played tracks, and no one played the untagged track.

### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [90]:
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

*Q5 gave me the most trouble. I originally used a left join rather than an inner join, but it overstated my calculated seconds and minutes becuase it included the length of unplayed tracks in the sum (that was supposed to only sum the length of the tracks played each time they were played). An inner join excluded the unplayed tracks since they didn't have a line in `plays`.*